# 07 · LoRA fine-tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/chronos2-in-practice/blob/main/notebooks/07_finetune_lora.ipynb)

Companion notebook for *Chronos-2 in Practice (4/5)*. **Needs a GPU** (local or Colab T4).

**What you will learn**

- A temporal split that cannot leak: train, validation and test periods in time order, with
  every validation window taking its context only from earlier data.
- `Chronos2Pipeline.fit(..., finetune_mode="lora")` end to end: adapter rank, learning rate,
  early stopping, saving and reloading.
- What one LoRA run costs — wall-clock time, peak VRAM, artifact size, trainable parameters.
- Whether rank 4, 8 or 16 matters, measured on the same 2018 windows as article 2.
- Three traps between `fit()` returning and a trustworthy evaluation, and why this repository
  reloads the checkpoint from disk instead of scoring the object `fit` hands back.

Runtime: ~30 min on an RTX 3070 (FAST_MODE: ~5 min).

In [ ]:
# On Google Colab, clone the companion repo and install it (Colab's own PyTorch is kept).
# Locally, `uv sync` already installed everything and this cell does nothing.
import sys

if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/YOUR_GITHUB_USER/chronos2-in-practice.git
    %cd chronos2-in-practice
    %pip install -q .
    import importlib

    importlib.invalidate_caches()

In [ ]:
import time

import pandas as pd
import torch
from huggingface_hub.utils import logging as hub_logging
from IPython.display import Image, display
from transformers.utils import logging as transformers_logging

from c2p.config import get_device, get_paths, is_fast_mode, set_seed
from c2p.data import spain_energy_frame, validate_and_prepare
from c2p.eval import backtest, evaluate, skill_table, summarize
from c2p.experiments import FREQ, LOAD, TRAIN_END, day_ahead_config
from c2p.finetune import (
    finetune,
    lora_config,
    temporal_split,
    to_fit_inputs,
    to_univariate_series,
    window_series,
)
from c2p.models import Chronos2Forecaster, load_chronos2
from c2p.plots.evaluation import plot_skill_scores
from c2p.plotting import save_figure, set_style
from c2p.results import load_results, run_metadata, save_results
from c2p.schema import ID_COL, TIMESTAMP_COL

FAST_MODE = is_fast_mode(default=False)
PATHS = get_paths(fast_mode=FAST_MODE)
set_seed()
DEVICE = get_device()
set_style()
hub_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

if DEVICE != "cuda":
    print(
        "WARNING: no GPU visible. This notebook fine-tunes a 120M-parameter model; on a CPU a\n"
        "single run takes hours. Open it in Colab (Runtime > Change runtime type > T4 GPU)."
    )
for key, value in run_metadata(device=DEVICE, fast_mode=FAST_MODE).items():
    print(f"{key:>20}: {value}")
if DEVICE == "cuda":
    print(
        f"{'gpu':>20}: {torch.cuda.get_device_name(0)} "
        f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB)"
    )

## 1. A split that cannot leak

Fine-tuning makes leakage much easier than zero-shot inference does, because the model now has
an opportunity to memorise. Three periods in time order are the minimum:

- **train** — everything up to mid-2017, used for the gradient updates;
- **validation** — the second half of 2017, used only for early stopping;
- **test** — 2018, the same year article 2 evaluated on, never seen in any form.

`window_series` is the detail that matters for the validation set: instead of handing the model
the validation period as one long series, it cuts it into forecast windows and makes each one a
separate series whose context comes from *earlier* data only. Validation loss then measures the
same thing as the test backtest, rather than the model's ability to continue a series it is
already inside.

There is also a data decision here. One national load series is thin training data, so every
numeric column of the Spanish table — load, price, and generation by source — becomes its own
univariate series with `to_univariate_series`. That is a domain fine-tune: "learn Spanish
electricity", not "learn this one series".

In [ ]:
HORIZON = 24
CONTEXT = 2048
STEPS = 50 if FAST_MODE else 1000
BATCH = 32

raw = spain_energy_frame()
numeric = [column for column in raw.columns if column not in (ID_COL, TIMESTAMP_COL)]
frame, report = validate_and_prepare(raw[[ID_COL, TIMESTAMP_COL, *numeric]], freq=FREQ, target=LOAD)
univariate = to_univariate_series(frame, numeric)
split = temporal_split(
    univariate, val_start=TRAIN_END + pd.Timedelta(hours=1), test_start="2018-01-01"
)
print(f"{univariate[ID_COL].nunique()} univariate series from the Spanish table")
print(
    f"train {split.train[TIMESTAMP_COL].min()} to {split.train[TIMESTAMP_COL].max()} "
    f"({len(split.train):,} rows)"
)
print(
    f"val   {split.val[TIMESTAMP_COL].min()} to {split.val[TIMESTAMP_COL].max()} "
    f"({len(split.val):,} rows)"
)
print(
    f"test  {split.test[TIMESTAMP_COL].min()} to {split.test[TIMESTAMP_COL].max()} "
    f"({len(split.test):,} rows)"
)

train_inputs = to_fit_inputs(split.train, freq=FREQ, prediction_length=HORIZON)
val_frame = window_series(
    split.train, split.val, prediction_length=HORIZON, freq=FREQ, step=168, max_context=CONTEXT
)
val_inputs = to_fit_inputs(val_frame, freq=FREQ, prediction_length=HORIZON)
print(f"\nfit inputs: {len(train_inputs)} training series, {len(val_inputs)} validation windows")

## 2. The reference to beat

Before training anything: the zero-shot number on the test windows. This is the same series,
the same 365 day-ahead windows and the same metrics as article 2, so if fine-tuning does not
beat this, that is the result.

In [ ]:
# The test series and protocol are exactly article 2's, so the numbers are comparable.
TEST_FRAME = frame[[ID_COL, TIMESTAMP_COL, LOAD]]
TEST_CONFIG = day_ahead_config(FAST_MODE, max_context=CONTEXT)
print(TEST_CONFIG)

zero_shot = Chronos2Forecaster(load_chronos2(device=DEVICE), name="zero-shot")
zero_metrics = evaluate(backtest(zero_shot, TEST_FRAME, TEST_CONFIG), TEST_FRAME, TEST_CONFIG)
print(f"zero-shot WQL on the 2018 test windows: {zero_metrics['WQL'].mean():.4f}")

## 3. One LoRA run, and what it costs

LoRA freezes the pretrained weights and trains two small matrices per targeted layer; the rank
`r` sets their width. The defaults here reproduce Chronos-2's own targets (the attention
projections and the output patch embedding) and AutoGluon's learning rate of 1e-5.

Three things happen between `fit()` returning and a number you can trust, and `c2p.finetune`
handles all three — they are the reason this notebook does not call `fit` directly:

1. Without validation data the returned pipeline stays in **training mode**, so dropout makes
   every forecast random.
2. After a full fine-tune with mixed precision the forward pass stays wrapped in **bfloat16
   autocast**, shifting forecasts by a fraction of a percent.
3. `fit` enables **TF32** process-wide on Ampere or newer GPUs and never switches it off, which
   silently changes every later forecast in the session.

`finetune()` returns the saved checkpoint reloaded from disk and restores TF32, so the model you
evaluate is the model you saved.

In [ ]:
RANKS = (8,) if FAST_MODE else (4, 8, 16)
runs = {}
rows = []
for rank in RANKS:
    result = finetune(
        load_chronos2(device=DEVICE),
        train_inputs,
        val_inputs=val_inputs,
        mode="lora",
        prediction_length=HORIZON,
        num_steps=STEPS,
        learning_rate=1e-5,
        batch_size=BATCH,
        context_length=CONTEXT,
        output_dir=PATHS.models / f"lora_r{rank}",
        lora_config=lora_config(r=rank),
        early_stopping_patience=3,
        eval_steps=10 if FAST_MODE else 100,
        overwrite=True,
    )
    runs[rank] = result
    rows.append(
        {
            "run": f"lora_r{rank}",
            "rank": rank,
            "trainable_parameters": result.trainable_parameters,
            "total_parameters": result.total_parameters,
            "train_seconds": result.train_seconds,
            "peak_vram_mb": result.peak_vram_mb,
            "artifact_mb": result.artifact_mb,
            "steps": result.steps,
            "best_step": result.best_step,
            "best_eval_loss": result.best_eval_loss,
        }
    )
    print(
        f"r={rank}: {result.trainable_parameters:,} trainable "
        f"({result.trainable_parameters / result.total_parameters:.2%} of the model), "
        f"{result.train_seconds / 60:.1f} min, peak VRAM {result.peak_vram_mb:.0f} MB, "
        f"adapter {result.artifact_mb:.1f} MB, best step {result.best_step}"
    )
lora_costs = pd.DataFrame(rows)
print("Saved", save_results(lora_costs, "07_lora_costs", PATHS).name)

## 4. Does it help?

Each adapter is evaluated on the test windows exactly as the zero-shot model was.

In [ ]:
start = time.perf_counter()
frames = [zero_metrics]
for rank, result in runs.items():
    model = Chronos2Forecaster(result.pipeline, name=f"lora_r{rank}")
    frames.append(evaluate(backtest(model, TEST_FRAME, TEST_CONFIG), TEST_FRAME, TEST_CONFIG))
    print(f"  lora_r{rank}: WQL {frames[-1]['WQL'].mean():.4f}")
lora_metrics = pd.concat(frames, ignore_index=True)
print(f"Evaluated in {(time.perf_counter() - start) / 60:.1f} min")
print("Saved", save_results(lora_metrics, "07_lora_metrics", PATHS).name)

In [ ]:
lora_metrics = load_results("07_lora_metrics", PATHS)
lora_skill = skill_table(lora_metrics, baseline="zero-shot", metric="WQL")
print("Saved", save_results(lora_skill, "07_lora_skill", PATHS).name)

best = lora_skill[lora_skill["model"] != "zero-shot"].iloc[0]
direction = "better" if best["skill"] > 0 else "WORSE"
title = (
    f"{best['model']} is {abs(best['skill']):.1%} {direction} than zero-shot "
    f"(95% CI [{best['ci_low']:+.1%}, {best['ci_high']:+.1%}])"
)
ax = plot_skill_scores(
    lora_skill,
    highlight=str(best["model"]),
    title=title,
    xlabel="WQL improvement over the zero-shot model (%)",
)
(figure_path,) = save_figure(ax.figure, "07_lora_skill", PATHS.figures)
display(Image(filename=figure_path, width=800))
lora_skill.round(4)

## 5. Summary

In [ ]:
costs = load_results("07_lora_costs", PATHS).set_index("run")
skill = load_results("07_lora_skill", PATHS).set_index("model")
summary = load_results("07_lora_metrics", PATHS)
means = summarize(summary).set_index("model")

print(f"{'run':<12} {'WQL':>8} {'vs zero-shot':>13} {'train':>8} {'VRAM':>8} {'adapter':>9}")
print(
    f"{'zero-shot':<12} {means.loc['zero-shot', 'WQL']:8.4f} {'--':>13} {'--':>8} "
    f"{'--':>8} {'--':>9}"
)
for run in costs.index:
    cost, sk = costs.loc[run], skill.loc[run]
    print(
        f"{run:<12} {means.loc[run, 'WQL']:8.4f} {sk['skill']:+12.1%} "
        f"{cost['train_seconds'] / 60:7.1f}m {cost['peak_vram_mb']:7.0f}M "
        f"{cost['artifact_mb']:8.1f}M"
    )
print(
    f"\nCoverage check (80% band): zero-shot {means.loc['zero-shot', 'coverage_80']:.1%}, "
    + ", ".join(f"{run} {means.loc[run, 'coverage_80']:.1%}" for run in costs.index)
)

**Next:** [`08_finetune_full`](08_finetune_full.ipynb) trains every weight instead of 1% of
them, and asks what that buys and what it breaks.